In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from essentials import resample_data, three_class_labels, two_class_labels, normalize
from features_acc_gyr_improved import GenerateFeatures 
import copy

In [2]:
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/train_df_normalized.pkl', 'rb') as f: 
    train_set_normalized_df = pickle.load(f)
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/test_df_normalized.pkl', 'rb') as f:
    test_set_normalized_df = pickle.load(f)
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/ground_truth_with_ids.pkl', 'rb') as f:
    ground_truth_with_ids = pickle.load(f)

In [3]:
groups_train = train_set_normalized_df['experiment_id'].unique()
groups_test = test_set_normalized_df['experiment_id'].unique()

In [4]:
# Separate the ground truth data into training and testing sets
ground_truth_train = ground_truth_with_ids[ground_truth_with_ids['experiment_id'].isin(groups_train)]
ground_truth_test = ground_truth_with_ids[ground_truth_with_ids['experiment_id'].isin(groups_test)]

In [6]:
# Create a dictionary of DataFrames for each experiment is the experiment id as the key and remove the 'experiment_id' column

train_dict = {exp_id: experiment_data for exp_id, experiment_data in train_set_normalized_df.groupby('experiment_id')}
test_dict = {exp_id: experiment_data for exp_id, experiment_data in test_set_normalized_df.groupby('experiment_id')}

ground_truth_train_dict = {exp_id: experiment_data for exp_id, experiment_data in ground_truth_train.groupby('experiment_id')}
ground_truth_test_dict = {exp_id: experiment_data for exp_id, experiment_data in ground_truth_test.groupby('experiment_id')}

In [7]:
# Remove the 'experiment_id' column from the ground truth DataFrames
for exp_id in ground_truth_train_dict.keys():
    ground_truth_train_dict[exp_id] = ground_truth_train_dict[exp_id].drop(columns=['experiment_id'])       
for exp_id in ground_truth_test_dict.keys():
    ground_truth_test_dict[exp_id] = ground_truth_test_dict[exp_id].drop(columns=['experiment_id'])     
    

In [8]:
# Remove the 'experiment_id' column from the train and test DataFrames
for exp_id in train_dict.keys():
    train_dict[exp_id] = train_dict[exp_id].drop(columns=['experiment_id'])
for exp_id in test_dict.keys():
    test_dict[exp_id] = test_dict[exp_id].drop(columns=['experiment_id'])

In [9]:
train_dict.keys()

dict_keys([1, 2, 3, 4, 6, 7, 8, 10, 11, 12, 13, 15, 16, 17, 18, 19, 21, 22, 23, 24, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37, 38, 39])

## Add 2 class labels 
1. non-void
2. void

In [10]:
labelled_train_dict = {}
dict = copy.deepcopy(train_dict)
dict_gt = copy.deepcopy(ground_truth_train_dict)

for i, exp_id in tqdm(enumerate(dict.keys()), desc="Adding labels to IMU data"):
    acc = dict[exp_id]
    gt = dict_gt[exp_id]
    
    # add the labels
    labelled_df = two_class_labels(acc, gt)
    
    labelled_train_dict[exp_id] = labelled_df

Adding labels to IMU data: 32it [00:00, 666.18it/s]


In [11]:
labelled_train_dict

{1:          acc_x     acc_y     acc_z     gyr_x     gyr_y     gyr_z       time  \
 0    -0.107579 -0.694253 -0.953834 -0.380594  0.135491  0.122457   0.000000   
 1    -0.138740 -0.706585 -0.940029 -0.258003  0.105213  0.119008   0.016935   
 2    -0.183801 -0.731279 -0.930357 -0.120522  0.079119  0.136082   0.033871   
 3    -0.208548 -0.749396 -0.925878 -0.036531  0.066959  0.201708   0.050806   
 4    -0.174317 -0.700010 -0.917477  0.047828  0.084565  0.297416   0.067741   
 ...        ...       ...       ...       ...       ...       ...        ...   
 2845 -0.657188 -0.717708 -0.962583 -0.266003  0.035480  0.021139  48.181074   
 2846 -0.632107 -0.714359 -0.955489 -0.238881  0.092311  0.046441  48.198010   
 2847 -0.603765 -0.693841 -0.951806 -0.151757  0.147046  0.071269  48.214945   
 2848 -0.596491 -0.687616 -0.956208 -0.127130  0.155795  0.081115  48.231880   
 2849 -0.609040 -0.704515 -0.964485 -0.199205  0.130758  0.096646  48.248816   
 
                       Real time   

In [13]:
labelled_test_dict = {}
dict = copy.deepcopy(test_dict)
dict_gt = copy.deepcopy(ground_truth_test_dict)

for i, exp_id in tqdm(enumerate(dict.keys()), desc="Adding labels to IMU data"):
    acc = dict[exp_id]
    gt = dict_gt[exp_id]
    
    # add the labels
    labelled_df = two_class_labels(acc, gt)
    
    labelled_test_dict[exp_id] = labelled_df

Adding labels to IMU data: 9it [00:00, 632.94it/s]


In [20]:
labelled_test_dict

{5:           acc_x     acc_y     acc_z     gyr_x     gyr_y     gyr_z       time  \
 12090 -0.445718 -0.700418 -0.885795  0.064583  0.103128  0.050092   0.000000   
 12091 -0.426920 -0.692952 -0.879153  0.012989  0.127989  0.035721   0.016931   
 12092 -0.440015 -0.671706 -0.861447  0.091577  0.159393  0.058052   0.033862   
 12093 -0.557381 -0.658361 -0.871835  0.177880  0.179420  0.184889   0.050793   
 12094 -0.590328 -0.711706 -0.896365  0.074363  0.222306  0.218905   0.067724   
 ...         ...       ...       ...       ...       ...       ...        ...   
 14185  0.188768 -0.747257 -1.031113  0.134709  0.053030 -0.035259  35.470393   
 14186  0.204987 -0.747041 -1.030551  0.103387  0.040565 -0.062248  35.487324   
 14187  0.210304 -0.740170 -1.029594  0.112720  0.044693 -0.076377  35.504255   
 14188  0.198600 -0.728956 -1.028293  0.121347  0.024356 -0.079139  35.521186   
 14189  0.189385 -0.723347 -1.026111  0.133810  0.022247 -0.086116  35.538117   
 
                       

In [14]:
# Save the dictionay
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/train_data_with_2_class_labels.pkl', 'wb') as f:
    pickle.dump(labelled_train_dict, f)
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/test_data_with_2_class_labels.pkl', 'wb') as f:
    pickle.dump(labelled_test_dict, f)

# Add 3 class labels
1. pre-void
2. void
3. post-void

In [15]:
labelled_train_dict_3 = {}
dict = copy.deepcopy(train_dict)
dict_gt = copy.deepcopy(ground_truth_train_dict)

for i, exp_id in tqdm(enumerate(dict.keys()), desc="Adding labels to IMU data"):
    acc = dict[exp_id]
    gt = dict_gt[exp_id]
    
    # add the labels
    labelled_df = three_class_labels(acc, gt)
    
    labelled_train_dict_3[exp_id] = labelled_df

Adding labels to IMU data: 32it [00:00, 579.29it/s]


In [16]:
labelled_train_dict_3

{1:          acc_x     acc_y     acc_z     gyr_x     gyr_y     gyr_z       time  \
 0    -0.107579 -0.694253 -0.953834 -0.380594  0.135491  0.122457   0.000000   
 1    -0.138740 -0.706585 -0.940029 -0.258003  0.105213  0.119008   0.016935   
 2    -0.183801 -0.731279 -0.930357 -0.120522  0.079119  0.136082   0.033871   
 3    -0.208548 -0.749396 -0.925878 -0.036531  0.066959  0.201708   0.050806   
 4    -0.174317 -0.700010 -0.917477  0.047828  0.084565  0.297416   0.067741   
 ...        ...       ...       ...       ...       ...       ...        ...   
 2845 -0.657188 -0.717708 -0.962583 -0.266003  0.035480  0.021139  48.181074   
 2846 -0.632107 -0.714359 -0.955489 -0.238881  0.092311  0.046441  48.198010   
 2847 -0.603765 -0.693841 -0.951806 -0.151757  0.147046  0.071269  48.214945   
 2848 -0.596491 -0.687616 -0.956208 -0.127130  0.155795  0.081115  48.231880   
 2849 -0.609040 -0.704515 -0.964485 -0.199205  0.130758  0.096646  48.248816   
 
                       Real time   

In [17]:
labelled_test_dict_3 = {}
dict = copy.deepcopy(test_dict)
dict_gt = copy.deepcopy(ground_truth_test_dict)

for i, exp_id in tqdm(enumerate(dict.keys()), desc="Adding labels to IMU data"):
    acc = dict[exp_id]
    gt = dict_gt[exp_id]
    
    # add the labels
    labelled_df = three_class_labels(acc, gt)
    
    labelled_test_dict_3[exp_id] = labelled_df

Adding labels to IMU data: 9it [00:00, 647.57it/s]


In [18]:
labelled_test_dict_3

{5:           acc_x     acc_y     acc_z     gyr_x     gyr_y     gyr_z       time  \
 12090 -0.445718 -0.700418 -0.885795  0.064583  0.103128  0.050092   0.000000   
 12091 -0.426920 -0.692952 -0.879153  0.012989  0.127989  0.035721   0.016931   
 12092 -0.440015 -0.671706 -0.861447  0.091577  0.159393  0.058052   0.033862   
 12093 -0.557381 -0.658361 -0.871835  0.177880  0.179420  0.184889   0.050793   
 12094 -0.590328 -0.711706 -0.896365  0.074363  0.222306  0.218905   0.067724   
 ...         ...       ...       ...       ...       ...       ...        ...   
 14185  0.188768 -0.747257 -1.031113  0.134709  0.053030 -0.035259  35.470393   
 14186  0.204987 -0.747041 -1.030551  0.103387  0.040565 -0.062248  35.487324   
 14187  0.210304 -0.740170 -1.029594  0.112720  0.044693 -0.076377  35.504255   
 14188  0.198600 -0.728956 -1.028293  0.121347  0.024356 -0.079139  35.521186   
 14189  0.189385 -0.723347 -1.026111  0.133810  0.022247 -0.086116  35.538117   
 
                       

In [19]:
# Save the dictionay
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/train_data_with_3_class_labels.pkl', 'wb') as f:
    pickle.dump(labelled_train_dict_3, f)
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/test_data_with_3_class_labels.pkl', 'wb') as f:
    pickle.dump(labelled_test_dict_3, f)